# REE Distribution Data Engineering - Qiwei Liang

 
**Scope:** Locality distribution of 17 rare-earth elements in the top 5 countries by rare-earth reserves.  

**Hypothesis1**:

Do global REE occurrences form distinct spatial clusters that correlate with specific tectonic plate boundaries?

**Hypothesis 2**:

Is there a significant difference in the proportion of light rare earth and heavy rare earth minerals among different clustering groups?

**Important note:**

1.Mindat locality data shows where elements are reported. It does not provide reliable ore grade, concentration percentage, or reserve tonnage by element. This notebook analyzes locality distribution, not chemical concentration.

2.The top 5 country reserve numbers are from USGS Mineral Commodity Summaries 2026, Rare Earths chapter.
https://pubs.usgs.gov/periodicals/mcs2025/mcs2025-rare-earths.pdf?utm

## Scope

This notebook focuses on the requirement: top 5 rare-earth reserve countries and 17 REE elements.

**Countries:** China, Brazil, Australia, Russia, Vietnam  
**Elements:** Sc, Y, La, Ce, Pr, Nd, Pm, Sm, Eu, Gd, Tb, Dy, Ho, Er, Tm, Yb, Lu




In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

# Store raw API responses separately from cleaned outputs.
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Mindat API key used by every request below.
API_KEY = "3b54da79accb60ccb08008b7271d668c"
HEADERS = {
    "Authorization": f"Token {API_KEY}",
    "Accept": "application/json",
    "User-Agent": "Qiwei-REE-project",
}

# High page size reduces pagination while staying simple for a class project notebook.
PAGE_SIZE = 500
SLEEP_SECONDS = 0.15

# USGS 2026 rare-earth reserves, in metric tons REO equivalent.
TOP_REE_COUNTRIES = pd.DataFrame([
    {"reserve_rank": 1, "country": "China", "reserves_reo_tonnes": 44_000_000},
    {"reserve_rank": 2, "country": "Brazil", "reserves_reo_tonnes": 21_000_000},
    {"reserve_rank": 3, "country": "Australia", "reserves_reo_tonnes": 6_300_000},
    {"reserve_rank": 4, "country": "Russia", "reserves_reo_tonnes": 3_800_000},
    {"reserve_rank": 5, "country": "Vietnam", "reserves_reo_tonnes": 3_500_000},
])

# Project definition of 17 REE elements: Sc + Y + 15 lanthanides.
REE_ELEMENTS = pd.DataFrame([
    {"symbol": "Sc", "element_name": "Scandium", "ree_group": "scandium"},
    {"symbol": "Y", "element_name": "Yttrium", "ree_group": "hree"},
    {"symbol": "La", "element_name": "Lanthanum", "ree_group": "lree"},
    {"symbol": "Ce", "element_name": "Cerium", "ree_group": "lree"},
    {"symbol": "Pr", "element_name": "Praseodymium", "ree_group": "lree"},
    {"symbol": "Nd", "element_name": "Neodymium", "ree_group": "lree"},
    {"symbol": "Pm", "element_name": "Promethium", "ree_group": "lree"},
    {"symbol": "Sm", "element_name": "Samarium", "ree_group": "lree"},
    {"symbol": "Eu", "element_name": "Europium", "ree_group": "lree"},
    {"symbol": "Gd", "element_name": "Gadolinium", "ree_group": "hree"},
    {"symbol": "Tb", "element_name": "Terbium", "ree_group": "hree"},
    {"symbol": "Dy", "element_name": "Dysprosium", "ree_group": "hree"},
    {"symbol": "Ho", "element_name": "Holmium", "ree_group": "hree"},
    {"symbol": "Er", "element_name": "Erbium", "ree_group": "hree"},
    {"symbol": "Tm", "element_name": "Thulium", "ree_group": "hree"},
    {"symbol": "Yb", "element_name": "Ytterbium", "ree_group": "hree"},
    {"symbol": "Lu", "element_name": "Lutetium", "ree_group": "hree"},
])

print("API key used:", API_KEY[:6] + "..." + API_KEY[-4:])
print("Countries:", TOP_REE_COUNTRIES["country"].tolist())
print("Elements:", REE_ELEMENTS["symbol"].tolist())


API key used: 3b54da...668c
Countries: ['China', 'Brazil', 'Australia', 'Russia', 'Vietnam']
Elements: ['Sc', 'Y', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu']


## Pull Data From Mindat

For each country and each REE element, the notebook pulls Mindat localities where that element is reported.



In [ ]:
def save_json(data, path):
     # Save data as a UTF-8 JSON file
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def fetch_localities(country, element):
    #Fetch Mindat localities for one country and one REE element.
    url = "https://api.mindat.org/v1/localities/"
    params = {
        "country": country,
        "elements_inc": element,
        "fields": "id,txt,country,latitude,longitude,elements,datemodify,parent,coordsystem",
        "page-size": PAGE_SIZE,
    }

    rows = []
    page = 1

    while url:
        response = requests.get(url, headers=HEADERS, params=params if page == 1 else None, timeout=60)

        # A 400 response can happen for unsupported or empty element filters; keep the run moving.
        if response.status_code == 400:
            print(f"{country} / {element}: API returned 400, saved as 0 rows")
            return []

        if response.status_code == 401:
            raise RuntimeError("Mindat API key was rejected. Check API_KEY.")

        response.raise_for_status()
        data = response.json()

        # Attach query metadata so each row keeps its country-element context after merging.
        for row in data.get("results", []):
            row["query_country"] = country
            row["query_element"] = element
            rows.append(row)

        url = data.get("next")
        page += 1
        time.sleep(SLEEP_SECONDS)

    return rows


In [ ]:
raw_rows = []

# Run 5 countries x 17 elements = 85 API queries.
for country in TOP_REE_COUNTRIES["country"]:
    for element in REE_ELEMENTS["symbol"]:
        rows = fetch_localities(country, element)
        raw_rows.extend(rows)
        print(f"{country} / {element}: {len(rows)} localities")

raw_path = RAW_DIR / "mindat_top5_country_17ree_localities_raw.json"
save_json(raw_rows, raw_path)

raw_df = pd.DataFrame(raw_rows)
print("Raw rows:", len(raw_df))
raw_df.head()


China / Sc: 30 localities
China / Y: 405 localities
China / La: 141 localities
China / Ce: 359 localities
China / Pr: 0 localities
China / Nd: 208 localities
China / Pm: 0 localities
China / Sm: 0 localities
China / Eu: 0 localities
China / Gd: 0 localities
China / Tb: 0 localities
China / Dy: 0 localities
China / Ho: 0 localities
China / Er: 0 localities
China / Tm: 0 localities
China / Yb: 14 localities
China / Lu: 0 localities
Brazil / Sc: 2 localities
Brazil / Y: 104 localities
Brazil / La: 41 localities
Brazil / Ce: 132 localities
Brazil / Pr: 0 localities
Brazil / Nd: 44 localities
Brazil / Pm: 0 localities
Brazil / Sm: 0 localities
Brazil / Eu: 0 localities
Brazil / Gd: 0 localities
Brazil / Tb: 0 localities
Brazil / Dy: 0 localities
Brazil / Ho: 0 localities
Brazil / Er: 0 localities
Brazil / Tm: 0 localities
Brazil / Yb: 0 localities
Brazil / Lu: 0 localities
Australia / Sc: 25 localities
Australia / Y: 283 localities
Australia / La: 111 localities
Australia / Ce: 267 localiti

,id,txt,latitude,longitude,datemodify,elements,country,coordsystem,parent,query_country,query_element
0,693,China,0.000000,0.000000,2024-11-26 07:44:08,-Si-O-Ag-S-Fe-Se-Ca-Mg-H-C-As-Zn-Al-K-Na-Ti-Ce...,China,0,0,China,Sc
1,701,"Guizhou, China",0.000000,0.000000,2025-09-20 18:13:10,-Ag-S-Ca-Fe-Mg-Si-O-H-As-Cu-Hg-Mn-Al-Na-Ti-Pb-...,China,0,693,China,Sc
2,705,"Hunan, China",0.000000,0.000000,2024-10-23 13:43:34,-Ag-S-Ca-Fe-Mg-Si-O-H-Al-K-Ce-Nb-Ti-Bi-Cu-Pb-M...,China,0,693,China,Sc
3,706,"Chenzhou, Hunan, China",0.000000,0.000000,2022-06-16 15:13:28,-Ag-S-Ca-Fe-Mg-Si-O-H-Bi-Cu-Pb-Mn-Al-Na-Te-K-T...,China,0,705,China,Sc
4,715,"Xianghualing Mine, Xianghualing Sn-polymetalli...",25.458611,112.567222,2026-01-25 16:21:47,-Ag-S-Ca-Fe-Mg-Si-O-H-Al-Na-K-As-Ba-Li-Be-Ti-F...,China,0,157020,China,Sc


## Clean Data

This section cleans raw Mindat locality data and builds the final tables for later analysis.


In [ ]:
def parse_elements(value):
    """Convert a Mindat element string like '-La-Ce-Nd-' into a Python list."""
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text.startswith("-") and text.endswith("-"):
        return [x for x in text.split("-") if x]
    return [x.strip() for x in text.split(",") if x.strip()]


clean_columns = [
    "reserve_rank", "query_country", "reserves_reo_tonnes", "query_element", "element_name", "ree_group",
    "locality_id", "locality_name", "country", "latitude", "longitude", "analysis_ready",
    "reported_elements", "datemodify", "parent", "coordsystem", "mindat_locality_url",
]

if raw_df.empty:
    clean = pd.DataFrame(columns=clean_columns)
else:
    clean = raw_df.rename(columns={"id": "locality_id", "txt": "locality_name"}).copy()
    clean["latitude"] = pd.to_numeric(clean["latitude"], errors="coerce")
    clean["longitude"] = pd.to_numeric(clean["longitude"], errors="coerce")
    clean["reported_elements"] = clean["elements"].map(parse_elements)

    # Keep rows with real map coordinates and remove the common placeholder coordinate (0, 0).
    valid_lat = clean["latitude"].between(-90, 90)
    valid_lon = clean["longitude"].between(-180, 180)
    not_zero = ~((clean["latitude"].fillna(0) == 0) & (clean["longitude"].fillna(0) == 0))
    clean["analysis_ready"] = valid_lat & valid_lon & not_zero

    clean["mindat_locality_url"] = clean["locality_id"].map(lambda x: f"https://www.mindat.org/loc-{int(x)}.html")

    # Add reserve-country metadata and element-group metadata to every locality row.
    country_info = TOP_REE_COUNTRIES.rename(columns={"country": "query_country"})
    element_info = REE_ELEMENTS.rename(columns={"symbol": "query_element"})
    clean = clean.merge(country_info, on="query_country", how="left")
    clean = clean.merge(element_info, on="query_element", how="left")
    clean = clean.drop_duplicates(subset=["query_country", "query_element", "locality_id"])
    clean = clean[[c for c in clean_columns if c in clean.columns]]

if clean.empty:
    analysis_ready = clean.copy()
else:
    analysis_ready = clean[clean["analysis_ready"]].copy()

# Long summary table: one row per country-element pair.
summary = (
    analysis_ready
    .groupby(["reserve_rank", "query_country", "reserves_reo_tonnes", "query_element", "element_name", "ree_group"], dropna=False)
    .agg(locality_count=("locality_id", "nunique"))
    .reset_index()
    .sort_values(["reserve_rank", "query_element"])
)

# Wide matrix table: countries as rows, elements as columns, locality counts as values.
matrix = summary.pivot_table(
    index="query_country",
    columns="query_element",
    values="locality_count",
    fill_value=0,
    aggfunc="sum",
)
matrix = matrix.reindex(index=TOP_REE_COUNTRIES["country"], columns=REE_ELEMENTS["symbol"], fill_value=0)

print("Clean rows:", len(clean))
print("Analysis-ready rows:", len(analysis_ready))
matrix


Clean rows: 3546
Analysis-ready rows: 1338


symbol,Sc,Y,La,Ce,Pr,Nd,Pm,Sm,Eu,Gd,Tb,Dy,Ho,Er,Tm,Yb,Lu
country,,,,,,,,,,,,,,,,,
China,6,114,40,109,0,63,0,0,0,0,0,0,0,0,0,3,0
Brazil,0,52,17,67,0,21,0,0,0,0,0,0,0,0,0,0,0
Australia,7,150,42,111,0,38,0,0,0,0,0,0,0,0,0,0,0
Russia,8,125,62,216,0,59,0,4,0,1,0,0,0,0,0,2,0
Vietnam,0,4,4,7,0,6,0,0,0,0,0,0,0,0,0,0,0


## Save Outputs

These are Qiwei's final data-engineering deliverables.


In [ ]:
clean_path = PROCESSED_DIR / "top5_17ree_localities_clean.csv"
analysis_path = PROCESSED_DIR / "top5_17ree_localities_analysis_ready.csv"
summary_path = PROCESSED_DIR / "top5_17ree_distribution_summary.csv"
matrix_path = PROCESSED_DIR / "top5_17ree_distribution_matrix.csv"
country_path = PROCESSED_DIR / "top5_ree_reserve_countries.csv"
elements_path = PROCESSED_DIR / "ree_17_elements.csv"
quality_path = PROCESSED_DIR / "top5_17ree_data_quality_report.json"

clean.to_csv(clean_path, index=False)
analysis_ready.to_csv(analysis_path, index=False)
summary.to_csv(summary_path, index=False)
matrix.to_csv(matrix_path)
TOP_REE_COUNTRIES.to_csv(country_path, index=False)
REE_ELEMENTS.to_csv(elements_path, index=False)

quality_report = {
    "scope": "Top 5 rare-earth reserve countries and 17 REE element locality distribution from Mindat.",
    "api_key_used_preview": API_KEY[:6] + "..." + API_KEY[-4:],
    "country_count": len(TOP_REE_COUNTRIES),
    "element_count": len(REE_ELEMENTS),
    "raw_rows": len(raw_df),
    "clean_rows": len(clean),
    "analysis_ready_rows": len(analysis_ready),
    "unique_localities": int(analysis_ready["locality_id"].nunique()) if not analysis_ready.empty else 0,
    "outputs": {
        "clean": str(clean_path),
        "analysis_ready": str(analysis_path),
        "summary": str(summary_path),
        "matrix": str(matrix_path),
        "countries": str(country_path),
        "elements": str(elements_path),
    },
}
save_json(quality_report, quality_path)

quality_report


{'scope': 'Top 5 rare-earth reserve countries and 17 REE element locality distribution from Mindat.',
 'api_key_used_preview': '3b54da...668c',
 'country_count': 5,
 'element_count': 17,
 'raw_rows': 3546,
 'clean_rows': 3546,
 'analysis_ready_rows': 1338,
 'unique_localities': 714,
 'outputs': {'clean': 'data\\processed\\top5_17ree_localities_clean.csv',
  'analysis_ready': 'data\\processed\\top5_17ree_localities_analysis_ready.csv',
  'summary': 'data\\processed\\top5_17ree_distribution_summary.csv',
  'matrix': 'data\\processed\\top5_17ree_distribution_matrix.csv',
  'countries': 'data\\processed\\top5_ree_reserve_countries.csv',
  'elements': 'data\\processed\\ree_17_elements.csv'}}

## Quick Checks

Basic checks before handing files to teammates.


In [ ]:
#Replace API_KEY with your own Mindat API key before running.
#You can apply for a key in minat.org, it will take a few days to get approved.
#But it's free for academic use and will allow you to run the notebook without hitting authentication errors.
assert API_KEY == "PASTE_YOUR_MINDAT_API_KEY_HERE"

assert len(TOP_REE_COUNTRIES) == 5
assert len(REE_ELEMENTS) == 17
assert set(TOP_REE_COUNTRIES["country"]) == {"China", "Brazil", "Australia", "Russia", "Vietnam"}
assert set(REE_ELEMENTS["symbol"]) == {"Sc", "Y", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"}
assert {"query_country", "query_element", "latitude", "longitude", "analysis_ready"}.issubset(clean.columns)

print("All checks passed.")
matrix


All checks passed.


symbol,Sc,Y,La,Ce,Pr,Nd,Pm,Sm,Eu,Gd,Tb,Dy,Ho,Er,Tm,Yb,Lu
country,,,,,,,,,,,,,,,,,
China,6,114,40,109,0,63,0,0,0,0,0,0,0,0,0,3,0
Brazil,0,52,17,67,0,21,0,0,0,0,0,0,0,0,0,0,0
Australia,7,150,42,111,0,38,0,0,0,0,0,0,0,0,0,0,0
Russia,8,125,62,216,0,59,0,4,0,1,0,0,0,0,0,2,0
Vietnam,0,4,4,7,0,6,0,0,0,0,0,0,0,0,0,0,0


---

# Downstream Team Workspace

Qiwei's data-engineering part ends here.

Recommended files for later members:

- `data/processed/top5_17ree_localities_analysis_ready.csv`
- `data/processed/top5_17ree_distribution_summary.csv`
- `data/processed/top5_17ree_distribution_matrix.csv`


## Mengna Cheng - GIS / Visualization

Blank workspace for maps and charts.


## Michael King - Clustering / ML

Blank workspace for clustering or modeling.


## Rohit Narwal - Technical Writing

Blank workspace for report notes and tables.


## Sources

- Local PDF: `DS _Assignment4a_Qiwei_Liang.pdf`
- Mindat API schema: `https://api.mindat.org/v1/schema/`
- USGS Mineral Commodity Summaries 2026, Rare Earths chapter
